# 🚢 Machine Learning Classification with the Titanic

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SharifiZarchi/IntroAI/blob/main/Session_06/Titanic_2/Titanic_Classification_Tutorial_2_EN.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FSharifiZarchi%2FIntroAI%2Fblob%2Fmain%2FSession_06%2FTitanic_2%2FTitanic_Classification_Tutorial_2_EN.ipynb)

On the night of April 15, 1912, the Titanic — "the unsinkable ship" — hit an iceberg. Of the roughly 2,200 people on board, more than 1,500 died. There were not enough lifeboats for everyone… so **who got a seat?**

That question became the most famous beginner competition on [Kaggle](https://www.kaggle.com/competitions/titanic): given a passenger's information (age, sex, ticket class, …), predict whether they survived. Today we solve it, step by step, and climb from a naive guess all the way to a solution worthy of the top of the leaderboard.

Last session we predicted a **number** (house prices) — that was **regression**. Today we predict a **category** (survived / died) — that is **classification**. Same recipe: load → look → clean → plot → model → evaluate.

**How to run:** click a grey cell and press `Shift + Enter`. Run cells in order, top to bottom. Each part ends with a small ✏️ exercise — try it before opening the solution!

Data from [Kaggle](https://www.kaggle.com/competitions/titanic), saved next to this file as `titanic.csv`.

---
# Part 1 — Warm-up: predicting a category

In the last notebook we met variables, lists, loops, and functions. (New here? Do Part 1 of the Session 4 notebook first — it takes five minutes.)

One Python block is still missing, and it happens to be the seed of today's main idea: `if / else` — **making a decision**.

### 1.1 `if / else` — the computer makes a choice
The computer checks a condition; if it is true it runs one block, otherwise another. Note the `:` and the indentation.

In [ ]:
age = 8

if age < 18:
    print("child")
else:
    print("adult")

### 1.2 A classifier written by hand
A **classifier** is just a function that takes features in and gives a category out. Here is a one-rule classifier for the Titanic. Keep it in mind — we will meet it again.

In [ ]:
def guess_survival(sex):
    if sex == "female":
        return 1     # survives
    else:
        return 0     # does not survive

print(guess_survival("female"))
print(guess_survival("male"))

### 1.3 Libraries
The same four friends as last time.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

print("Libraries ready ✅")

### ✏️ Exercise 1
Change the code below so it prints three groups: `"child"` for under 13, `"teenager"` for 13 to 17, and `"adult"` for 18 and above. (Hint: `elif` means "else, if…")

In [ ]:
age = 15

# ✏️ Your turn — add an elif for teenagers
if age < 13:
    print("child")
else:
    print("adult")

<details>
<summary>💡 Solution — click to open</summary>

```python
age = 15

if age < 13:
    print("child")
elif age < 18:
    print("teenager")
else:
    print("adult")
```

</details>

---
# Part 2 — Load the data

One row per passenger, one column per piece of information.

In [ ]:
# The file sits next to this notebook; if it is not found (e.g. on Google Colab),
# we read the exact same file from the course's GitHub page.
url = "https://raw.githubusercontent.com/SharifiZarchi/IntroAI/main/Session_05/Titanic/titanic.csv"
try:
    df = pd.read_csv("titanic.csv")
except FileNotFoundError:
    df = pd.read_csv(url)

print("Our table has", df.shape[0], "rows and", df.shape[1], "columns")

See the first few rows with `.head()`:

In [ ]:
df.head()

What each column means:

| Column | Meaning |
|---|---|
| `Survived` | **the label we want to predict**: 1 = survived, 0 = died |
| `Pclass` | ticket class: 1 = first (expensive), 2 = middle, 3 = third (cheap) |
| `Name`, `Sex`, `Age` | name, sex, age |
| `SibSp` | number of siblings + spouses on board |
| `Parch` | number of parents + children on board |
| `Ticket`, `Fare` | ticket number and ticket price |
| `Cabin` | cabin number (if known) |
| `Embarked` | boarding port: S = Southampton, C = Cherbourg, Q = Queenstown |

In Session 3 language: `Survived` is the **label**, everything else is a **feature**.

### ✏️ Exercise 2
`.head()` shows the first rows; `.tail()` shows the last ones. Show the last 5 passengers of the table.

In [ ]:
# ✏️ Your turn



<details>
<summary>💡 Solution — click to open</summary>

```python
df.tail()
```

</details>

---
# Part 3 — Look at the data

Same three commands as last time: `.info()`, `.describe()`, `.value_counts()`.

### 3.1 `.info()` — column types and missing values

In [ ]:
df.info()

The table has 891 rows, but `Age` has only 714 values — the age of 177 passengers is unknown. `Cabin` is almost empty (687 missing!) and `Embarked` misses 2. Real data is never complete; we deal with this in Part 5.

### 3.2 `.describe()` — quick stats

In [ ]:
df.describe()

Read the `mean` row: the average of `Survived` is about 0.38 — **only 38% survived**. The average age is about 30, and ticket prices run from 0 to a spectacular 512 pounds.

### 3.3 `.value_counts()` — count categories

In [ ]:
print("Survived?")
print(df["Survived"].value_counts())

print("\nSex:")
print(df["Sex"].value_counts())

print("\nTicket class:")
print(df["Pclass"].value_counts())

549 died, 342 survived. Most passengers were men, and most travelled third class. Remember these numbers — they set the stage.

### ✏️ Exercise 3
How many passengers boarded at each port? Use `.value_counts()` on the `Embarked` column.

In [ ]:
# ✏️ Your turn



<details>
<summary>💡 Solution — click to open</summary>

```python
print(df["Embarked"].value_counts())
# S (Southampton) 644, C (Cherbourg) 168, Q (Queenstown) 77
```

</details>

---
# Part 4 — Insight: see the answer before modeling

This is the most important part of the notebook. Before any model, we ask the data: **who survived?** Every pattern we spot here is something a good model should also find — so when we build models later, we can check whether they "understood" the data.

### 4.1 Survival by sex
`.groupby("Sex")` splits the passengers into groups, and `.mean()` of a 0/1 column is exactly the **survival rate**.

In [ ]:
rate_by_sex = df.groupby("Sex")["Survived"].mean()
print(rate_by_sex)

plt.figure(figsize=(5, 4))
plt.bar(["female", "male"], rate_by_sex[["female", "male"]], color=["mediumseagreen", "steelblue"])
plt.ylabel("Survival rate")
plt.title("Survival by sex")
plt.ylim(0, 1)
plt.show()

A huge gap: **74% of women survived, versus 19% of men.** "Women and children first" was not just a saying — it is right here in the data, a century later. This single fact is the strongest signal in the whole dataset.

### 4.2 Survival by ticket class

In [ ]:
rate_by_class = df.groupby("Pclass")["Survived"].mean()
print(rate_by_class)

plt.figure(figsize=(5, 4))
plt.bar(["1st", "2nd", "3rd"], rate_by_class[[1, 2, 3]], color="coral")
plt.ylabel("Survival rate")
plt.title("Survival by ticket class")
plt.ylim(0, 1)
plt.show()

First class 63%, second 47%, third 24%. The lifeboats were on the upper decks, next to the expensive cabins. Money mattered.

### 4.3 And the children?
Two overlapping histograms: ages of those who survived (green) and those who died (grey).

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df[df["Survived"] == 0]["Age"].dropna(), bins=30, alpha=0.6, color="gray", label="Died")
plt.hist(df[df["Survived"] == 1]["Age"].dropna(), bins=30, alpha=0.6, color="mediumseagreen", label="Survived")
plt.xlabel("Age")
plt.ylabel("Count")
plt.title("Age distribution: survived vs died")
plt.legend()
plt.show()

print("Survival rate of children under 10:", round(df[df["Age"] < 10]["Survived"].mean(), 2))
print("Survival rate of everyone:         ", round(df["Survived"].mean(), 2))

Look at the far left of the chart: for young children the green bars beat the grey ones — **61% of children under 10 survived**, versus 38% overall. The "children" part of "women and children first" checks out too.

### 4.4 Sex and class together
The two strongest signals combined in one picture.

In [ ]:
table = df.groupby(["Pclass", "Sex"])["Survived"].mean().unstack()
print(table.round(2))

table.plot(kind="bar", figsize=(7, 4), color=["mediumseagreen", "steelblue"])
plt.ylabel("Survival rate")
plt.title("Survival by class and sex")
plt.xticks([0, 1, 2], ["1st", "2nd", "3rd"], rotation=0)
plt.ylim(0, 1)
plt.legend(["female", "male"])
plt.show()

The extremes are stunning: a first-class woman survived with **97%** probability; a third-class man with **13%**. Between these two passengers lies the whole story of the Titanic — and everything our models are about to learn.

### 4.5 What a classification problem looks like
One last picture, the most important one conceptually. Each dot is a passenger: age on one axis, fare on the other. Green = survived, grey = died.

In [ ]:
plt.figure(figsize=(8, 5))
colors = df["Survived"].map({0: "gray", 1: "mediumseagreen"})
plt.scatter(df["Age"], df["Fare"], c=colors, alpha=0.5, s=20)
plt.xlabel("Age")
plt.ylabel("Fare")
plt.ylim(0, 150)   # a few very expensive tickets are cut off so we can see the rest
plt.title("Each dot is a passenger (green = survived)")
plt.show()

**This picture *is* classification.** A classifier's entire job: given a new dot's position, guess its color. You can see a tendency — green gathers at high fares and young ages — but the colors are mixed, and no single straight line can separate them. That is exactly why we need models smarter than a ruler.

(Remember the alien-classification slide from Session 2? Same picture, real data.)

### ✏️ Exercise 4
Draw the survival rate by boarding port (`Embarked`) as a bar chart, like we did for sex. Which port had the luckiest passengers? Any guess why? (Hint from 4.2: think about ticket class.)

In [ ]:
# ✏️ Your turn
# rate_by_port = df.groupby(...)[...].mean()



<details>
<summary>💡 Solution — click to open</summary>

```python
rate_by_port = df.groupby("Embarked")["Survived"].mean()
print(rate_by_port)

plt.figure(figsize=(5, 4))
plt.bar(["C", "Q", "S"], rate_by_port[["C", "Q", "S"]], color="mediumpurple")
plt.ylabel("Survival rate")
plt.title("Survival by boarding port")
plt.show()
```

Cherbourg (55%) beats Queenstown (39%) and Southampton (34%) — mostly because many first-class passengers boarded at Cherbourg. The port itself did not save anyone; it is a hidden echo of ticket class!

</details>

---
# Part 5 — Clean and prepare

Models need a table of **numbers with no holes**. We have holes (missing ages) and text (`Sex`, `Embarked`). Three fixes, same tools as last session.

### 5.1 Fill the holes
- Missing `Age` → fill with the median age (28).
- 2 missing `Embarked` → fill with the most common port (`S`).
- `Cabin` is missing for 687 of 891 passengers — too many holes to fill honestly, so we simply won't use it (for now… it returns in Part 9!).
- One honest footnote: we compute that median from all 891 rows *before* splitting; strictly it should come from the training rows only, so nothing leaks from the test set. (Here both give the same 28 — real pipelines fit the imputer on the training data alone.)


In [ ]:
data = df.copy()          # work on a copy, keep the original safe

data["Age"] = data["Age"].fillna(data["Age"].median())
data["Embarked"] = data["Embarked"].fillna("S")

print("Missing values now:")
print(data[["Age", "Embarked"]].isna().sum())

### 5.2 Turn text into numbers
- `Sex` → a new 0/1 column `Female` (this trick works for any two-category column).
- `Embarked` has three categories → one-hot encoding with `get_dummies`, exactly like the neighborhoods last session.

In [ ]:
data["Female"] = (data["Sex"] == "female").astype(int)
data = pd.get_dummies(data, columns=["Embarked"], prefix="Port")

data[["Name", "Sex", "Female", "Port_C", "Port_Q", "Port_S"]].head()

### 5.3 Features, label, and the train/test split
Same ritual as always: the model learns on the training part and is graded on the test part it has never seen.

In [ ]:
FEATURES = ["Pclass", "Female", "Age", "SibSp", "Parch", "Fare",
            "Port_C", "Port_Q", "Port_S"]

X = data[FEATURES]        # features (inputs)
y = data["Survived"]      # label (what we predict)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42   # 20% for testing
)

print("Train:", len(X_train), " Test:", len(X_test))

### ✏️ Exercise 5
Make sure our features really have no holes left: print the number of missing values in each column of `X`. (Hint: `.isna().sum()`)

In [ ]:
# ✏️ Your turn



<details>
<summary>💡 Solution — click to open</summary>

```python
print(X.isna().sum())
# every number should be 0
```

</details>

---
# Part 6 — First models: from a blind guess to a decision tree

How do we grade a classifier? With **accuracy**: the share of passengers it labels correctly. 1.0 means perfect, and… well, let's see what "zero effort" earns.

### 6.1 The laziest model in the world
It always says "died", no matter who you are. Sounds dumb — but since most passengers did die, it is right more often than not! We will collect every model's score in a `results` dictionary.

In [ ]:
from sklearn.metrics import accuracy_score

pred_lazy = np.zeros(len(y_test))          # 0 = "died", for everyone
acc = accuracy_score(y_test, pred_lazy)

results = {}                               # scoreboard for the whole session
results["Always 'died'"] = acc
print("Accuracy:", round(acc, 3))

**59% while knowing nothing!** This is called a **baseline**, and it teaches a crucial lesson: never be impressed by an accuracy number until you know what a blind guess scores. A "90% accurate" model for a disease that 95% of people don't have is *worse* than doing nothing.

### 6.2 One rule beats no rules
Now our hand-made classifier from Part 1: women survive, men don't. One `if`, based on the strongest pattern from Part 4.

In [ ]:
pred_one_rule = X_test["Female"]           # 1 for women, 0 for men — exactly our prediction
acc = accuracy_score(y_test, pred_one_rule)

results["One rule: sex"] = acc
print("Accuracy:", round(acc, 3))

**78% with a single if!** That jump — from 59 to 78 — came entirely from *insight*, not machinery. But squeezing out the next few percent by hand (rules for class? age? combinations?) gets messy fast. Time to let the machine write the rules.

### 6.3 The decision tree — a machine that learns to play "20 questions"
A **decision tree** asks yes/no questions about the features, one after another, and each answer leads to the next question — a tree of `if / else` blocks. The magic of *learning*: the machine itself discovers, from the training data, **which questions to ask and in what order** (it picks the question that best splits survivors from victims, then repeats inside each branch).

`max_depth=3` means: at most 3 questions per passenger.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)                 # learning happens here

acc = tree.score(X_test, y_test)           # .score = predict + accuracy in one step
results["Decision tree"] = acc
print("Accuracy:", round(acc, 3))

### 6.4 Look inside the tree's head 🌳
Unlike most models, a tree can show us exactly how it thinks.

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(16, 8))
plot_tree(tree, feature_names=FEATURES, class_names=["Died", "Survived"],
          filled=True, rounded=True, fontsize=9)
plt.show()

Read the top box: the tree's very first question is **"Female ≤ 0.5?"** — is this passenger a man? It rediscovered our hand-made rule *on its own*, then refined it: for men it asks about age (young boys get a chance), for women it asks about class and fare. Orange boxes lean "died", blue lean "survived".

Every pattern we saw in the plots of Part 4 — sex, class, children — is here, found automatically. **The model agrees with our eyes.**

### 6.5 Deeper is better… right?
More questions = more precision, surely. Let's push the depth up and watch both scores.

In [ ]:
depths = range(1, 15)
train_acc, test_acc = [], []

for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    train_acc.append(t.score(X_train, y_train))
    test_acc.append(t.score(X_test, y_test))

plt.figure(figsize=(8, 4.5))
plt.plot(depths, train_acc, marker="o", label="Train accuracy")
plt.plot(depths, test_acc, marker="o", label="Test accuracy")
plt.xlabel("Tree depth (number of questions)")
plt.ylabel("Accuracy")
plt.title("Memorizing vs learning")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

There it is — the most famous picture in machine learning, drawn from our own data. Train accuracy climbs toward 95%: a deep tree asks so many questions it effectively memorizes *every single passenger*. But test accuracy stops improving and slips back. That growing gap is **overfitting** — exactly the concept from Sessions 3 and 4, now witnessed live. A depth of 3–5 is the sweet spot here.

### ✏️ Exercise 6
Train a tree with `max_depth=1` — a "stump" that may ask only one question — and print its test accuracy. Compare it with the one-rule model from 6.2. Surprised?

In [ ]:
# ✏️ Your turn
# stump = DecisionTreeClassifier(max_depth=..., random_state=42)



<details>
<summary>💡 Solution — click to open</summary>

```python
stump = DecisionTreeClassifier(max_depth=1, random_state=42)
stump.fit(X_train, y_train)
print("Stump accuracy:", round(stump.score(X_test, y_test), 3))
```

It scores exactly the same as our hand-made sex rule — because with only one question allowed, the best question to ask *is* "is this passenger female?". The machine and our intuition agree.

</details>

---
# Part 7 — KNN: you are like your neighbors

A completely different philosophy. **K-Nearest-Neighbors** learns no rules at all. To predict a new passenger, it simply finds the *k* most similar passengers in the training data — the "nearest neighbors" — and lets them **vote**. A 25-year-old woman with a first-class ticket? Find the 5 most similar passengers; if most of them survived, predict "survives".

"Similar" means *close* in the picture from Part 4.5 — just with all 9 features instead of 2.

### 7.1 First attempt

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

print("Accuracy:", round(knn.score(X_test, y_test), 3))

**71%?! Worse than our one-line rule!** What went wrong?

Think about what "close" means. `Fare` runs from 0 to 512, while `Female` is just 0 or 1. When computing distance, a 30-pound fare difference completely drowns out the difference between a man and a woman — the most important feature we have! KNN was judging similarity almost entirely by ticket price.

**The fix:** put every feature on the same scale first. `StandardScaler` does exactly that, and a `Pipeline` chains scaler + model so they act as one (both familiar from last session).

### 7.2 Second attempt: scale first

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

knn_scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
knn_scaled.fit(X_train, y_train)

acc = knn_scaled.score(X_test, y_test)
results["KNN (scaled)"] = acc
print("Accuracy:", round(acc, 3))

From 71% to 80% — **the model didn't change, the data's clothing did.** Rule of thumb: any model based on distances (like KNN) needs scaled features. Trees don't care, because "Age ≤ 6.5?" works the same on any scale.

### 7.3 Watch KNN think
A bonus picture — don't worry about this code, enjoy the result. We train a small KNN on just two features (age and fare) so we can paint its decisions: every point of the plane is colored by what the model *would* predict there.

In [ ]:
from sklearn.inspection import DecisionBoundaryDisplay

two_features = X_train[["Age", "Fare"]][X_train["Fare"] < 150]
two_labels = y_train[two_features.index]

knn_2d = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15))
knn_2d.fit(two_features.values, two_labels)

display = DecisionBoundaryDisplay.from_estimator(
    knn_2d, two_features.values, response_method="predict",
    grid_resolution=300, alpha=0.25, cmap="RdYlGn")
display.ax_.scatter(two_features["Age"], two_features["Fare"],
                    c=two_labels.map({0: "gray", 1: "green"}), s=15, alpha=0.7)
plt.xlabel("Age")
plt.ylabel("Fare")
plt.title("KNN's mind: green regions = predicts 'survives'")
plt.show()

This is the answer to Part 4.5's challenge: the model drew the boundary — and it is no straight line. Wherever a new dot lands, its color is decided by the region it falls in. Islands of green form around clusters of survivors: *you are like your neighbors.*

### 7.4 Choosing k — the same old trade-off
`k` is the number of voting neighbors. Small k: every prediction copies one or two passengers (memorization). Huge k: everyone gets the same answer (oversimplification). Sound familiar? It is tree depth all over again — in reverse.

In [ ]:
ks = [1, 3, 5, 7, 9, 11, 15, 21, 31, 51]
train_acc, test_acc = [], []

for k in ks:
    p = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)).fit(X_train, y_train)
    train_acc.append(p.score(X_train, y_train))
    test_acc.append(p.score(X_test, y_test))

plt.figure(figsize=(8, 4.5))
plt.plot(ks, train_acc, marker="o", label="Train accuracy")
plt.plot(ks, test_acc, marker="o", label="Test accuracy")
plt.xlabel("k (number of neighbors)")
plt.ylabel("Accuracy")
plt.title("Choosing k")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

At `k=1` train accuracy is a spectacular 96% — of course: each training passenger's nearest neighbor is *themselves*. Pure memorization, and the test score exposes it. Around k = 5–7 the test score peaks. `k` and `max_depth` are **hyperparameters**: knobs *we* set, which the model does not learn itself (Session 3 vocabulary!).

One caution: we just picked `k` by peeking at the *test* scores — do that too often and the test set stops being a fair judge. The professional fix is **cross-validation** (Part 9.6).


### ✏️ Exercise 7
Predict for *yourself*! Fill in your own values below (would you have travelled 1st, 2nd or 3rd class? 😉) and see what the scaled KNN predicts for you.

In [ ]:
# ✏️ Your turn — replace the values with your own
me = pd.DataFrame({
    "Pclass": [3],     # your ticket class: 1, 2 or 3
    "Female": [0],     # 1 = female, 0 = male
    "Age": [25],       # your age
    "SibSp": [0],      # siblings/spouse on board
    "Parch": [0],      # parents/children on board
    "Fare": [15],      # ticket price (3rd ≈ 14, 2nd ≈ 21, 1st ≈ 84 pounds)
    "Port_C": [0], "Port_Q": [0], "Port_S": [1],
})

# print("Prediction:", knn_scaled.predict(me)[0], "  (1 = survives)")

<details>
<summary>💡 Solution — click to open</summary>

```python
print("Prediction:", knn_scaled.predict(me)[0], "  (1 = survives)")

# You can also ask for the vote itself — the estimated probability:
print("Chance of survival:", knn_scaled.predict_proba(me)[0][1])
```

`predict_proba` shows the vote count: e.g. 0.4 means 2 of the 5 neighbors survived.

</details>

---
# Part 8 — Random forest: the wisdom of the crowd

A single tree is like a single expert: smart, but opinionated — change the training data slightly and you may get a very different tree. The fix is more than a century old: in 1906, statistician Francis Galton watched a village fair where ~800 people guessed an ox's weight. Individual guesses were all over the place; their **average missed by less than 1%**.

A **random forest** = hundreds of trees, each trained on a random sample of the passengers and, at each question it asks, allowed to look at only a random subset of the features (so the trees don't all copy the strongest feature and end up identical — their *mistakes* stay independent). To predict: all trees vote.

### 8.1 Train the forest
`n_estimators` = number of trees. We keep each tree modest (`max_depth=5`) — Part 6 taught us why.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
forest.fit(X_train, y_train)

acc = forest.score(X_test, y_test)
results["Random forest"] = acc
print("Accuracy:", round(acc, 3))

Our best score yet — the crowd beats every single expert we've tried.

### 8.2 What does the forest consider important?
We can't draw 200 trees, but the forest can tell us how much each feature contributed to its decisions.

In [ ]:
importances = pd.Series(forest.feature_importances_, index=FEATURES).sort_values()

plt.figure(figsize=(8, 4.5))
plt.barh(importances.index, importances.values, color="seagreen")
plt.xlabel("Importance")
plt.title("What the forest pays attention to")
plt.show()

Sex on top, then fare and age — the forest independently confirms everything our plots said in Part 4. This loop — *see a pattern in the data, then check the model found it too* — is how you build trust in a model.

### ✏️ Exercise 8
Is a bigger crowd always wiser? Train forests with `n_estimators=5` and `n_estimators=500` and compare their test accuracies to our 200-tree forest.

In [ ]:
# ✏️ Your turn



<details>
<summary>💡 Solution — click to open</summary>

```python
for n in [5, 200, 500]:
    f = RandomForestClassifier(n_estimators=n, max_depth=5, random_state=42)
    f.fit(X_train, y_train)
    print(n, "trees:", round(f.score(X_test, y_test), 3))
```

5 trees is a bit weaker; 200 → 500 changes nothing at all. The crowd's wisdom saturates — after a point, more trees only cost more time.

</details>

---
# Part 9 — The champion's recipe: a top-Kaggle solution

Time for the summit. First, an honest warning: on the Kaggle leaderboard you will see scores of 100%. **They are cheating** — the Titanic's real passenger list is public, so answers can be looked up. Honest top solutions score around **80–83%**, and here is their real secret:

> It is *not* a fancier model. It is **feature engineering** — using human insight to build better features.

Which is great news for us: it means the path to the top runs through exactly the skill we practiced in Part 4 — *understanding the data*. We'll craft four new features, then bring in one final model.

### 9.1 The treasure hidden in names
We have ignored the `Name` column so far. Look closely:

In [ ]:
print(df["Name"].head(3).tolist())

Every name contains a **title** — the word between the comma and the period: `Mr`, `Mrs`, `Miss`, `Master`… Let's extract it (`str.extract` grabs exactly that middle piece) and keep the four common ones, grouping the rare ones (`Dr`, `Rev`, `Countess`…) as `"Rare"`.

In [ ]:
data["Title"] = df["Name"].str.extract(r",\s*([^.]+)\.")[0].str.strip()
data["Title"] = data["Title"].replace(["Mlle", "Ms"], "Miss").replace("Mme", "Mrs")
data["Title"] = data["Title"].where(data["Title"].isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")

print(data["Title"].value_counts())
print()
print("Survival rate by title:")
print(data.groupby("Title")["Survived"].mean().round(2))

Why is this gold? In 1912, **"Master" was the title for young boys** — so this feature tells the model who the children are *even when age is missing*! And look at the survival rates: `Mr` 16%, `Master` 58%, `Mrs` 79%. One extracted word carries sex, age, and social status all at once.

### 9.2 Family size
`SibSp` and `Parch` were weak features on their own (the forest barely used them). But their *sum* — the whole family travelling together — tells a story:

In [ ]:
data["FamilySize"] = data["SibSp"] + data["Parch"] + 1     # +1 = the passenger themself
data["IsAlone"] = (data["FamilySize"] == 1).astype(int)

rate_by_family = data.groupby("FamilySize")["Survived"].mean()
plt.figure(figsize=(7, 4))
plt.bar(rate_by_family.index.astype(str), rate_by_family.values, color="steelblue")
plt.xlabel("Family size")
plt.ylabel("Survival rate")
plt.title("Survival by family size")
plt.show()

Travelling alone: 30%. Small family (2–4): 55–72% — families helped each other reach the boats, and women with children got priority. Large family (5+): the rate collapses — big families were mostly poor third-class passengers who searched for each other as the ship went down. **Two "useless" columns, added together, became a story.**

### 9.3 A smarter fix for missing ages
In Part 5 we filled every missing age with 28 — including for boys titled "Master"! Now we can do better: fill each passenger's missing age with the median of *their own title group*.

In [ ]:
print("Median age per title:")
print(df.groupby(data["Title"])["Age"].median())

data["Age"] = df["Age"]                     # restore the original ages (with holes)
data["Age"] = data.groupby("Title")["Age"].transform(lambda ages: ages.fillna(ages.median()))

print("\nMissing ages now:", data["Age"].isna().sum())

A missing-age "Master" now gets 3.5 years instead of 28 — a child, as he almost surely was. Cleaning done with *insight* instead of a blanket rule.

### 9.4 Even the holes carry information
We threw `Cabin` away because it was 77% empty. But *whether the cabin was recorded at all* speaks:

In [ ]:
data["HasCabin"] = df["Cabin"].notna().astype(int)

print(data.groupby("HasCabin")["Survived"].mean().round(2))

Passengers with a recorded cabin survived at 67% vs 30% — recorded cabins mostly belonged to the upper decks. Sometimes **missingness itself is a feature**.

### 9.5 The final model: gradient boosting
Our new feature table, plus one last idea. A random forest builds its trees *independently*, in parallel. **Gradient boosting** builds them *in sequence*: each new tree is trained specifically on **the mistakes of the team so far**, nudging predictions a small step toward the truth.

That "small step toward less error" should ring a bell — it is **gradient descent**, the connecting thread of this course, now building trees. The step size is even called `learning_rate`, exactly like in Session 4. This family of models (and its tuned cousins XGBoost / LightGBM) wins more real tabular-data competitions than anything else.

In [ ]:
data = pd.get_dummies(data, columns=["Title"], prefix="Title")

FEATURES_2 = ["Pclass", "Female", "Age", "SibSp", "Parch", "Fare",
              "Port_C", "Port_Q", "Port_S",
              "FamilySize", "IsAlone", "HasCabin",
              "Title_Mr", "Title_Mrs", "Title_Miss", "Title_Master", "Title_Rare"]

X2 = data[FEATURES_2]
y2 = data["Survived"]
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

from sklearn.ensemble import GradientBoostingClassifier
boost = GradientBoostingClassifier(random_state=42)
boost.fit(X2_train, y2_train)

acc = boost.score(X2_test, y2_test)
results["Boosting + features"] = acc
print("Accuracy:", round(acc, 3))

**84%** — our best score, and honestly earned. Note that most of the gain came from the *features*, not the model: the same boosting on the old features scores about like the forest.

### 9.6 Measure like a professional: cross-validation
One 80/20 split means our score depends on which 179 passengers happened to land in the test set — luck. **Cross-validation** removes the luck: split the data into 5 parts, train 5 times, each time testing on a different part, and average the 5 scores.

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(GradientBoostingClassifier(random_state=42), X2, y2, cv=5)

print("5 scores:", scores.round(3))
print("Mean:", round(scores.mean(), 3), "±", round(scores.std(), 3))

About **83% ± 2%** — a trustworthy estimate, squarely in honest top-Kaggle territory. When you compare models seriously, always prefer this number to a single split.

### 9.7 The whole session in one picture

In [ ]:
plt.figure(figsize=(8, 4.5))
names = list(results.keys())
scores = [results[n] for n in names]
colors = ["gray"] + ["steelblue"] * (len(names) - 2) + ["mediumseagreen"]

bars = plt.barh(names, scores, color=colors)
for bar, s in zip(bars, scores):
    plt.text(s + 0.005, bar.get_y() + bar.get_height() / 2, f"{s:.3f}", va="center")
plt.xlim(0.5, 0.95)
plt.xlabel("Accuracy on the test set")
plt.title("Our climb: from a blind guess to the top")
plt.show()

From 59% to 84%. Read the bars bottom-to-top and notice *where* the big jumps came from: one insight about sex (+19), machine-learned rules (+2), fixing the scaling, the crowd (+2), and human-crafted features (+3). **Insight did more work than any algorithm.**

### 9.8 Your ticket to Kaggle 🎫
You are ready for a real competition. On [kaggle.com/competitions/titanic](https://www.kaggle.com/competitions/titanic), you get `train.csv` (with answers — what we used today) and `test.csv` (418 passengers, answers hidden). Train on the first, predict the second, and upload a two-column file:

```python
predictions = boost.predict(X_kaggle_test)
submission = pd.DataFrame({"PassengerId": test_ids, "Survived": predictions})
submission.to_csv("submission.csv", index=False)
```

Kaggle grades you instantly and places you on the leaderboard. With today's notebook you should score close to 0.80 — better than most first submissions in history. Ignore the 1.00 "winners"; you know their secret now.

### ✏️ Exercise 9 — the final challenge
Earn your own +0.4%: create the feature `FarePerPerson = Fare / FamilySize` (a family shares one ticket price!), add it to `FEATURES_2`, and re-run the cross-validation from 9.6. Did the mean improve?

In [ ]:
# ✏️ Your turn
# data["FarePerPerson"] = ...



<details>
<summary>💡 Solution — click to open</summary>

```python
data["FarePerPerson"] = data["Fare"] / data["FamilySize"]

X3 = data[FEATURES_2 + ["FarePerPerson"]]
scores = cross_val_score(GradientBoostingClassifier(random_state=42), X3, y2, cv=5)
print("Mean:", round(scores.mean(), 3), "±", round(scores.std(), 3))
```

The mean rises from about 0.832 to about 0.837. Small — but on Kaggle leaderboards, battles are won by exactly such steps. Now invent a feature of your own!

</details>

---
# Wrap-up

Today, on one real dataset: what classification is, baselines and **accuracy**, hand-made rules, **decision trees** (and reading their minds), overfitting witnessed live, **KNN** and why scaling matters, hyperparameters, **random forests** and feature importance, **feature engineering** (titles, family size, informative missingness), **gradient boosting**, and **cross-validation**.

Two lessons above all:

1. **Insight beats machinery.** Every big jump on our scoreboard came from understanding people on a ship in 1912.
2. **Trust models you can question.** We checked every model against the patterns our own eyes found in Part 4.

Your homework: make a Kaggle account, submit, and send us your score. Sink the leaderboard. 🚢🎉